# StockSense Pro — Exploratory Data Analysis & System Architecture
### CRS Artificial Intelligence Summative Assessment: Machine Learning and Deep Learning (FA 2)

**Student Name:** Aditya Jitendra Kumar Sahani  
**Student ID:** 1000414  
**Institution:** Aspee Nutan Academy  
**Programme:** IBCP Artificial Intelligence / Grade 12  
**Course:** Machine Learning and Deep Learning (CRS FA 2 / IADAI-201)  
**Project:** StockSense Pro: AI-Powered Retail Shelf Intelligence & Product Detection System  

---

## 1. Problem Formulation & Research Objectives
In physical retail environments, manual shelf auditing is labor-intensive, error-prone, and causes delayed stock replenishment. Studies indicate that on-shelf stockouts cost global retailers over \$1 trillion annually in lost sales and diminished customer loyalty.

**StockSense Pro** investigates whether an edge-deployable object-detection architecture (YOLOv8) fine-tuned on supermarket shelving scenes can:
1. Accurately detect and classify products across standardized grocery categories.
2. Count visible front-facing inventory items per category under varying lighting and clutter.
3. Translate visual facing counts into operational inventory alerts (*Out of Stock*, *Low Visible Stock*, *Sufficient Stock*).

In [ ]:
import os
import sys
import json
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# Add repository root to path
repo_root = Path("..").resolve()
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

from config import settings
print(f"StockSense Pro initialized for student: {settings.STUDENT_NAME} (ID: {settings.STUDENT_ID})")

## 2. Mandatory Primary Dataset Investigation: Kaggle Retail Product Checkout (RPC)
As instructed in Section 3 of the assessment brief, we first inspect the mandatory dataset:  
**Kaggle Retail Product Checkout (RPC) Dataset** (`diyer22/retail-product-checkout-dataset`).

In [ ]:
from src.dataset_analysis import analyze_rpc_dataset

rpc_path = repo_root / "data_investigation" / "rpc" / "instances_val2019.json"
rpc_stats = analyze_rpc_dataset(rpc_path)

print("=== Kaggle RPC Dataset Structural Findings ===")
print(f"Total Fine-Grained Categories: {rpc_stats['total_categories']}")
print(f"Total Macro Product Families: {rpc_stats['macro_categories_count']}")
print(f"Images in Validation Split:   {rpc_stats['total_images_in_split']}")
print(f"Bounding Box Annotations:     {rpc_stats['total_annotations_in_split']}")
print(f"Camera Perspective:           {rpc_stats['camera_viewpoint']}")
print(f"License:                      {rpc_stats['license']}")

### Scientific Critique: Why RPC is Insufficient for Shelf Monitoring Alone
Our investigation revealed that the RPC dataset was created specifically for **Automatic Checkout (ACO)**:
- Images are taken from a **top-down nadir angle** over a flat white checkout tray.
- Products lay horizontally or in jumbled piles on the tray.
- Retail shelf monitoring, however, requires a **frontal viewpoint** capturing upright bottles, multi-tier horizontal shelves, price rails, and vertical facing depths.

Therefore, as permitted and guided by Section 3 of the brief, we maintain RPC for comparative exploratory research and introduce a supplementary **Multi-Tier Retail Shelf Detection Benchmark** for model training.

In [ ]:
# Display Camera Perspective Comparison Diagram
comp_img_path = repo_root / "assets" / "eda" / "checkout_vs_shelf_comparison.png"
if comp_img_path.exists():
    display(Image.open(comp_img_path))

## 3. Supplementary Benchmark: Retail Shelf Detection Dataset
We investigate the supplementary Retail Shelf benchmark containing multi-tier retail shelving units across standardized grocery categories.

In [ ]:
from src.dataset_analysis import analyze_shelf_dataset

shelf_dir = repo_root / "data_investigation" / "shelf_dataset" / "extracted"
shelf_stats, aspect_ratios = analyze_shelf_dataset(shelf_dir)

print("=== Retail Shelf Benchmark Dataset Findings ===")
print(f"Total Files in Archive:       {shelf_stats['total_files']}")
print(f"Unique Base Shelf Scenes:     {shelf_stats['unique_base_scenes']}")
print(f"Total Standardized Boxes:     {shelf_stats['total_standardized_annotations']:,}")
print("\nGround Truth Distribution across Standardized Categories:")
for cname, cnt in shelf_stats["standardized_counts"].items():
    print(f"  - {cname:12s}: {cnt:,} annotated facings")

## 4. Preprocessing & Leakage Prevention
To prevent data leakage, we group files by their parent scene ID before splitting.
This guarantees that no augmented variant of a shelf scene appears in both training and test splits.

In [ ]:
summary_json = repo_root / "data" / "processed" / "split_summary.json"
if summary_json.exists():
    with open(summary_json) as f:
        split_data = json.load(f)
    print("Leak-Free Dataset Splits:")
    print(f"Train Scenes: {split_data['train_scenes']} (70%)")
    print(f"Val Scenes:   {split_data['val_scenes']} (15%)")
    print(f"Test Scenes:  {split_data['test_scenes']} (15%)")
    for split, s in split_data["stats"].items():
        print(f"  {split.upper()}: {s['images']} images, {s['boxes']:,} bounding boxes")

## 5. Visual Inspection of Ground Truth Annotations
Let us inspect a ground-truth annotated retail shelf from the benchmark to verify bounding box precision.

In [ ]:
sample_annotated = repo_root / "assets" / "eda" / "shelf_annotated_sample.png"
if sample_annotated.exists():
    img = Image.open(sample_annotated)
    print("Verified Annotated Retail Shelf Image:", img.size)
    display(img)

## 6. Inventory Intelligence Rules Demonstration
StockSense Pro translates visible instance counts into real-time operational alerts.

In [ ]:
from src.inventory_logic import InventoryEngine

engine = InventoryEngine()
test_detections = [
    {"class_name": "Bottle", "confidence": 0.91, "bbox": [10, 20, 50, 100]},
    {"class_name": "Bottle", "confidence": 0.88, "bbox": [55, 20, 95, 100]},
    {"class_name": "Can", "confidence": 0.79, "bbox": [100, 20, 140, 70]},
    {"class_name": "Can", "confidence": 0.84, "bbox": [145, 20, 185, 70]},
    {"class_name": "Can", "confidence": 0.77, "bbox": [190, 20, 230, 70]},
    {"class_name": "Can", "confidence": 0.82, "bbox": [235, 20, 275, 70]},
    {"class_name": "Snack", "confidence": 0.65, "bbox": [280, 20, 320, 80]},
]

eval_res = engine.evaluate_inventory(test_detections)
df_res = engine.export_to_dataframe(eval_res)
print("Inventory Intelligence Audit Table:")
display(df_res)